# **import Packages and data**

In [2]:
!pip install category_encoders

     |████████████████████████████████| 81kB 5.0MB/s 


In [3]:
from sklearn import metrics
from sklearn import metrics
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelBinarizer, LabelEncoder
from category_encoders import *
from datetime import datetime

from sklearn.metrics import roc_auc_score
from sklearn.metrics import log_loss
from sklearn.metrics import f1_score

/usr/local/lib/python3.6/dist-packages/statsmodels/tools/_testing.py:19: FutureWarning: pandas.util.testing is deprecated. Use the functions in the public API at pandas.testing instead.
  import pandas.util.testing as tm


In [4]:
drive.mount('/content/drive')
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

Mounted at /content/drive


In [66]:
train = pd.read_csv("/content/drive/MyDrive/ML_Project/train_data.csv")
test = pd.read_csv("/content/drive/MyDrive/ML_Project/test_data.csv")
y_train = train['clicked']
train = train.drop('clicked', 1)

,displayId,timestamp,dayOfWeek,hourOfDay,advertiserId,campaignId,creativeId,publisher,widgetId,device,os,browser,source,docId,userId
0,4706262,1578429005696,4,0,290,7855,6,10,6262,0,0,0,11,3543873,2688642


In [68]:
train = train.drop('displayId', 1)
train = train.drop('device', 1)
train = train.drop('docId', 1)
train = train.drop('userId', 1)
train = train.drop('timestamp', 1)

test = test.drop('displayId', 1)
test = test.drop('device', 1)
test = test.drop('docId', 1)
test = test.drop('userId', 1)
test = test.drop('timestamp', 1)
print(train.shape)
print(test.shape)

(3768416, 10)
(1072876, 10)


In [6]:
def reporter(fm, X, Y):
    pred = fm.predict(X)
    preda = fm.predict_proba(X)
    print(f'AUC: {roc_auc_score(Y, pred)}')
    print(f'F1: {f1_score (Y, pred)}')
    print(f'cross entropy: { log_loss (Y, preda)}')

# **FM MODEL**

In [66]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
import time

class FactorizationMachineClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_iter = 10, n_factors = 10,
                 learning_rate = 0.1, reg_coef = 0.01,
                 reg_factors = 0.01, random_state = 1234):
        self.n_iter = n_iter
        self.reg_coef = reg_coef
        self.n_factors = n_factors
        self.reg_factors = reg_factors
        self.random_state = random_state
        self.learning_rate = learning_rate

    def fit(self, X, y):

        n_samples, n_features = X.shape
        self.coef_ = np.zeros(n_features)
        self.intercept_ = 0.0
        np.random.seed(self.random_state)
        self.feature_factors_ = np.random.normal(
            scale = 1 / np.sqrt(self.n_factors), size = (self.n_factors, n_features))
        y = y.copy().astype(np.int32)
        y[y == 0] = -1

        
        self.history_ = []
        for iter in range(self.n_iter):
            loss = _sgd_update(X.data, X.indptr, X.indices,
                               y, n_samples, n_features,
                               self.intercept_, self.coef_,
                               self.feature_factors_, self.n_factors,
                               self.learning_rate, self.reg_coef, self.reg_factors)   
            # self.learning_rate = 0.9 * self.learning_rate
            self.history_.append(loss)
            print(f'iter {iter}, loss: {loss}')
        return self

    def predict_proba(self, X):
        pred = self._predict(X)
        pred_proba = 1.0 / (1.0 + np.exp(-pred))
        proba = np.vstack((1 - pred_proba, pred_proba)).T
        return proba

    def _predict(self, X):
        linear_output = X * self.coef_
        v = self.feature_factors_.T
        term = (X * v) ** 2 - (X.power(2) * (v ** 2))
        factor_output = 0.5 * np.sum(term, axis = 1)
        return self.intercept_ + linear_output + factor_output

    def predict(self, X):
        pred_proba = self.predict_proba(X)[:, 1]
        return pred_proba.round().astype(np.int)


def _sgd_update(data, indptr, indices, y, n_samples, n_features,
                w0, w, v, n_factors, learning_rate, reg_w, reg_v):
    loss = 0.0
    start = time.time()
    sample_size = n_samples
    for i in np.random.permutation(n_samples)[:sample_size]:
        pred, summed = _predict_instance(data, indptr, indices, w0, w, v, n_factors, i)
        loss += _log_loss(pred, y[i])
        loss_gradient = -y[i] / (np.exp(y[i] * pred) + 1.0)
        w0 -= learning_rate * loss_gradient
        for index in range(indptr[i], indptr[i + 1]):
            feature = indices[index]
            w[feature] -= learning_rate * (loss_gradient * data[index] + 2 * reg_w * w[feature])
        for factor in range(n_factors):
            for index in range(indptr[i], indptr[i + 1]):
                feature = indices[index]
                term = summed[factor] - v[factor, feature] * data[index]
                v_gradient = loss_gradient * data[index] * term
                v[factor, feature] -= learning_rate * (v_gradient + 2 * reg_v * v[factor, feature])
    loss /= n_samples
    return loss

def _predict_instance(data, indptr, indices, w0, w, v, n_factors, i):
    summed = np.zeros(n_factors)
    summed_squared = np.zeros(n_factors)
    pred = w0
    for index in range(indptr[i], indptr[i + 1]):
        feature = indices[index]
        pred += w[feature] * data[index]
    for factor in range(n_factors):
        for index in range(indptr[i], indptr[i + 1]):
            feature = indices[index]
            term = v[factor, feature] * data[index]
            summed[factor] += term
            summed_squared[factor] += term * term
        pred += 0.5 * (summed[factor] * summed[factor] - summed_squared[factor])
    return pred, summed

def _log_loss(pred, y):
    return np.log(np.exp(-pred * y) + 1.0)

# **Preparing Data**

In [7]:
train = train.drop('displayId', 1)
train = train.drop('device', 1)
train = train.drop('docId', 1)
train = train.drop('userId', 1)
train = train.drop('timestamp', 1)

test = test.drop('displayId', 1)
test = test.drop('device', 1)
test = test.drop('docId', 1)
test = test.drop('userId', 1)
test = test.drop('timestamp', 1)
print(train.shape)
print(test.shape)

(3768416, 10)
(1072876, 10)


In [188]:
from sklearn.preprocessing import OneHotEncoder
enc = OneHotEncoder()
enc.fit(pd.concat((train,test),0).to_numpy())

OneHotEncoder(categories='auto', drop=None, dtype=<class 'numpy.float64'>,
              handle_unknown='error', sparse=True)

In [189]:
train_one_hot = enc.transform(train)

In [179]:
TRAIN_SIZE = 400000

In [180]:
df_majority = train[y_train == 0]
df_minority = train[y_train == 1]
df_majority = resample(df_majority, 
                                 replace=True,     
                                 n_samples=int(TRAIN_SIZE / 2),    
                                 random_state=123) 

df_minority = resample(df_minority, 
                                 replace=True,     
                                 n_samples=int(TRAIN_SIZE / 2),    
                                 random_state=123) 

df_train = pd.concat([df_majority, df_minority])
Y_train = y_train[df_train.index].to_numpy()
df_train = enc.transform(df_train)

In [181]:
permu = np.random.permutation(TRAIN_SIZE)
X,Y = df_train[permu], Y_train[permu]

In [ ]:
t = pd.concat((train,test),0)

In [187]:
print([len(t[a].unique()) for a in t.columns])

[7, 24, 213, 528, 3256, 782, 1209, 8, 80, 2893]


In [191]:
feature_sizes = np.array([7, 24, 213, 528, 3256, 782, 1209, 8, 80, 2893])

# **TRAIN-FM**

In [86]:
start = time.time()
fm = FactorizationMachineClassifier(n_iter = 10, n_factors = 10, learning_rate = 0.005)
fm.fit(X, Y)
print(time.time() - start)

iter 0, loss: 0.6840245826094701
iter 1, loss: 0.6566780373818596
iter 2, loss: 0.6516581601821275
iter 3, loss: 0.6489019489722191
iter 4, loss: 0.6468384706280447
iter 5, loss: 0.6451400451595504
iter 6, loss: 0.6437326128507012
iter 7, loss: 0.6425813035739608
iter 8, loss: 0.6416509564396325
iter 9, loss: 0.6406369958857036
1762.9930319786072


In [87]:
##TRAIN
metrics.confusion_matrix(Y, fm.predict(X), normalize='all')

array([[0.2719725, 0.2280275],
       [0.138825 , 0.361175 ]])

In [88]:
###TRAIN
reporter (fm, X, Y)

AUC: 0.6331475000000001
F1: 0.663191647099598
cross entropy: 0.6381402466293098


In [89]:
##ALL
metrics.confusion_matrix(y_train.to_numpy(), fm.predict(train_one_hot), normalize='all')

array([[0.41009777, 0.36641814],
       [0.06632203, 0.15716205]])

In [90]:
##ALL
reporter (fm, train_one_hot, y_train.to_numpy())

AUC: 0.615680696252346
F1: 0.4207457315498558
cross entropy: 0.6910691037520853


In [91]:
import dill
filename = '/content/drive/MyDrive/ML_Project/FM/fm.pickle'
dill.dump(fm, open(filename, 'wb'))

In [93]:
fm2 = dill.load(open(filename, 'rb'))

In [95]:
test_one_hot = enc.transform(test)

In [100]:
final_pred = fm2.predict(test_one_hot)

In [101]:
np.save('/content/drive/MyDrive/ML_Project/FM/pred',final_pred)

# **Report-FM**

<div dir= 'rtl'>
برای این قسمت در ابتدا مقاله‌ مطالعه شد. برای شروع توجه کنید، دادگانی که در اختیار داریم mutli-categorical هستند و لذا در این نوع دادگان بعد از اعمال one_hot  یک تنکی شدیدی وجود دارد. در این نوع‌ دادگان روش‌های مرسومی که در قسمت‌های قبلی در موردشان صحبت شد،‌ نمی‌توانند کارا باشند. اولا توجه کنید interaction بین جفت ویژگی‌ها تأثیر بسزایی دارند و عموما در این نوع توزیع دادگان همه‌ی حالات جفت ویژگی‌ها در دادگان آموزش ظاهر نمی‌شوند و لذا مدل‌های مرسوم نظیر svm نمی‌توانند کارا باشند. روش FM با توجه به embedding که برای هر ویژگی انجام می‌دهد، باعث می‌شود که حتی اگر حالتی از نمونه‌ها را در طی آموزش ندیده باشد، بتواند که در مرحله ارزیابی پیش‌بینی خوبی انجام دهد. از این رو این روش و روش‌های مشابه می‌توانند کارایی خوبی روی این نوع دادگان داشته باشند.

برای استفاده از FM نیاز بود که دادگان آماده شوند. برای آماده‌سازی بر طبق نتایج EDA تعدادی از ویژگی‌ها را حذف کردیم. در ادامه تبدیل one_hot را روی دادگان انجام دادیم. با توجه به اینکه دادگان imbalanced بودن و همچنین خود روش FM طول می‌کشید. به صورت تصادفی از هر دسته تعدادی ( هایپر پارامتر)  مساوی را انتخاب کردیم و لذا تا این مرحله دادگان برای آموزش آماده می‌شوند.

توجه کنید که دادگانی که در اختیار داریم تنک هستند و اگر توسط ساختار داده خاصی استفاده نشود مشکلات نظیر پر شدن رم رخ می‌دهد. خروجی one_hot توسط کتابخانه sklearn این امکان را برای‌ ما فراهم می‌سازد و در نهایت کد 
الگوریتم FM طوری پیاده‌سازی کردیم که با  دادگان تنک نیز برخورد مناسبی داشته باشد و مشکلی از لحاظ منابع رخ ندهد. از این رو کد زده شده این نکته را رعایت می‌کند و به خوبی آموزش انجام می‌شود.

با توجه به اینکه ما از تعدادی از داده‌ها برای یادگیری استفاده می‌کنیم باید از بیش‌برازش مواظبت کنیم و لذا از هموارسازی درجه ۲ برای همه وزن‌ها نیز استفاده کردیم. اما باز هم پارامتر‌های دیگری داریم مثل  بعد  )embedding تعداد          (factorداریم و اگر این بعد مقدار زیادی باشد،‌ باز هم بیش‌برازش رخ می‌دهد و لذا این مقدار را نیز خیلی زیاد نکردیم تا مشکلاتی این‌چنینی رخ ندهد. همچنین با توجه به نوع پیاده‌سازی و اینکه سایز هر batch برابر با یک است. نیاز داریم که learning-rate هم مقداری زیاد نباشد و لذا با انتخاب چند نوع learning-rate به این مقدار رسیدیم. اما توجه کنید که در طول آموزش سعی می‌کنیم که این مقدار را نیز کم کنیم تا به نقطه بهتری همگرا شویم.

در مورد نتیجه نیز همانطور که انتظار می‌رفت کمی بهبود حاصل شد و مقادیر قابل مقایسه هستند. اما خب بازهم خیلی مقادیر بهتر نشدند.




</div>

# **FWFM**

In [182]:
import os,sys,random
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from time import time

import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.autograd import Variable

import torch.backends.cudnn


"""
    Network structure
"""

class FWFM(torch.nn.Module):
  
    def __init__(self,field_size, feature_sizes, embedding_size = 4, is_shallow_dropout = True, dropout_shallow = [0.0,0.0],
                 h_depth = 3,eval_metric = roc_auc_score, n_epochs = 64, batch_size = 2048, learning_rate = 0.001, momentum = 0.9,
                 is_batch_norm = False, verbose = False, random_seed = 0, weight_decay = 0.0,
                 use_fm = False, use_fwlw = False, use_lw = True, use_ffm = False, use_fwfm= False,loss_type = 'logloss',
                 use_cuda = True, n_class = 2, greater_is_better = True, sparse = 0.9, warm = 10, num_deeps = 1, numerical=0, use_logit=0
                 ):
        super(FWFM, self).__init__()
        self.field_size = field_size
        self.feature_sizes = feature_sizes
        self.embedding_size = embedding_size
        self.is_shallow_dropout = is_shallow_dropout
        self.dropout_shallow = dropout_shallow
        self.h_depth = h_depth
        self.n_epochs = n_epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.momentum = momentum
        self.is_batch_norm = is_batch_norm
        self.verbose = verbose
        self.weight_decay = weight_decay
        self.random_seed = random_seed
        self.use_fm = use_fm
        self.use_fwlw = use_fwlw
        self.use_lw = use_lw
        self.use_ffm = use_ffm
        self.use_fwfm = use_fwfm
        self.use_logit = use_logit
        self.loss_type = loss_type
        self.eval_metric = eval_metric
        self.use_cuda = use_cuda
        self.n_class = n_class
        self.greater_is_better = greater_is_better
        self.target_sparse = sparse
        self.warm = warm
        self.num = numerical
        np.random.seed(self.random_seed)
        random.seed(self.random_seed)
        torch.manual_seed(self.random_seed)
        torch.cuda.manual_seed(self.random_seed)


        self.use_deep = False
        """
            check cuda
        """
        if self.use_cuda and not torch.cuda.is_available():
            self.use_cuda = False
            print("Cuda is not available, automatically changed into cpu model")
        """
            check use fm, fwfm or ffm
        """
        if int(self.use_fm) + int(self.use_ffm) + int(self.use_fwfm) + int(self.use_logit) > 1:
            print("only support one type only, please make sure to choose only LR, FM, FFM or FwFM part")
            exit(1)
        elif self.use_logit:
            print("The model is logistic regression.")
        elif self.use_fm and self.use_deep:
            print("The model is deepfm(fm+deep layers)")
        elif self.use_ffm and self.use_deep:
            print("The model is deepffm(ffm+deep layers)")
        elif self.use_fwfm and self.use_deep:
            print("The model is deepfwfm(fwfm+deep layers)")
        elif self.use_fm:
            print("The model is fm only")
        elif self.use_ffm:
            print("The model is ffm only")
        elif self.use_fwfm:
            print("The model is fwfm only")
        elif self.use_deep:
            print("The model is deep layers only")
        else:
            print("You have to choose more than one of (fm, ffm, fwfm, deep) models to use")
            exit(1)

        """
            bias
        """
        if self.use_logit or self.use_fm or self.use_ffm or self.use_fwfm:
            self.bias = torch.nn.Parameter(torch.Tensor([0.01]))
        """
            LR/fm/fwfm part
        """
        if self.use_logit or self.use_fm or self.use_fwfm:
            if self.use_logit:
                print("Init Losgistic regression")
            elif self.use_fm:
                print("Init fm part")
            else:
                print("Init fwfm part")
            if not self.use_fwlw:
                self.fm_1st_embeddings = nn.ModuleList([nn.Embedding(feature_size,1) for feature_size in self.feature_sizes])
            if self.dropout_shallow:
                self.fm_first_order_dropout = nn.Dropout(self.dropout_shallow[0])
            if self.use_fm or self.use_fwfm:
                self.fm_2nd_embeddings = nn.ModuleList([nn.Embedding(feature_size, self.embedding_size) for feature_size in self.feature_sizes])
                if self.dropout_shallow:
                    self.fm_second_order_dropout = nn.Dropout(self.dropout_shallow[1])
            
                if (self.use_fm or self.use_fwfm or self.use_ffm) and self.use_lw:
                    self.fm_1st = nn.Linear(self.field_size, 1, bias=False)

                if (self.use_fm or self.use_fwfm or self.use_ffm) and self.use_fwlw:
                    self.fwfm_linear = nn.Linear(self.embedding_size, self.field_size, bias=False)
            
                if self.use_fwfm:
                    self.field_cov = nn.Linear(field_size, field_size, bias=False)

    def forward(self, Xi, Xv):
        """
        :param Xi_train: index input tensor, batch_size * embedding_size * 1
        :return: the last output
        """
        """
            fm/fwfm part
        """
        t00 = time()
        if self.use_logit or self.use_fm or self.use_fwfm:
            # dim: embedding_size * batch * 1, time cost 47%
            Tzero = torch.zeros(Xi.shape[0], 1, dtype=torch.long)
            if self.use_cuda:
                Tzero = Tzero.cuda()
            if not self.use_fwlw:
                fm_1st_emb_arr = [torch.sum(emb(Xi[:,i-self.num,:]),1) \
                        for i, emb in enumerate(self.fm_1st_embeddings)]
                # dim: batch_size * field_size
                fm_first_order = torch.cat(fm_1st_emb_arr, 1)
                if self.is_shallow_dropout:
                    fm_first_order = self.fm_first_order_dropout(fm_first_order)
                #print(fm_first_order.shape, "old linear")
            # dim: field_size * batch_size * embedding_size, time cost 43%
            if self.use_fm or self.use_fwfm:
                fm_2nd_emb_arr = [torch.sum(emb(Xi[:,i-self.num,:]),1) \
                        for i, emb in enumerate(self.fm_2nd_embeddings)]
                # convert a list of tensors to tensor
                fm_second_order_tensor = torch.stack(fm_2nd_emb_arr)
                if self.use_fwlw:
                    fwfm_linear = torch.einsum('ijk,ik->ijk', [fm_second_order_tensor, self.fwfm_linear.weight])
                    fm_first_order = torch.einsum('ijk->ji', [fwfm_linear])
                    if self.is_shallow_dropout:
                        fm_first_order = self.fm_first_order_dropout(fm_first_order)
                    #print(fm_first_order.shape, "new fwfm linear")

                outer_fm = torch.einsum('kij,lij->klij', fm_second_order_tensor, fm_second_order_tensor)
                if self.use_fm:
                    fm_second_order = (torch.sum(torch.sum(outer_fm, 0), 0) - torch.sum(torch.einsum('kkij->kij', outer_fm), 0)) * 0.5
                else:
                    # time cost 3%
                    outer_fwfm = torch.einsum('klij,kl->klij', outer_fm, (self.field_cov.weight.t() + self.field_cov.weight) * 0.5)
                    fm_second_order = (torch.sum(torch.sum(outer_fwfm, 0), 0) - torch.sum(torch.einsum('kkij->kij', outer_fwfm), 0)) * 0.5
                if self.is_shallow_dropout:
                    fm_second_order = self.fm_second_order_dropout(fm_second_order)

        """
            sum
        """
        # total_sum dim: batch, time cost 1.3%
        if (self.use_fm or self.use_fwfm) and self.use_lw:
            fm_first_order = torch.matmul(fm_first_order, self.fm_1st.weight.t())
        elif self.use_ffm and self.lw:
            ffm_first_order = torch.matmul(ffm_first_order, self.ffm_1st.weight.t())

        total_sum = torch.sum(fm_first_order, 1) + torch.sum(fm_second_order, 1) + self.bias
        return total_sum

    # credit to https://github.com/ChenglongChen/tensorflow-DeepFM/blob/master/DeepFM.py
    def init_weights(self):
        model = self.train()
        require_update = True
        last_layer_size = 0
        TORCH = torch.cuda if self.use_cuda else torch
        for name, param in model.named_parameters():
            if '1st_embeddings' in name:
                param.data = TORCH.FloatTensor(param.data.size()).normal_()
            elif '2nd_embeddings' in name:
                param.data = TORCH.FloatTensor(param.data.size()).normal_().mul(0.01)
            elif 'linear' in name:
                if 'weight' in name: # weight and bias in the same layer share the same glorot
                    glorot =  np.sqrt(2.0 / np.sum(param.data.shape))
                param.data = TORCH.FloatTensor(param.data.size()).normal_().mul(glorot)
            elif 'field_cov.weight' == name:
                param.data = TORCH.FloatTensor(param.data.size()).normal_().mul(np.sqrt(2.0 / self.field_size / 2))
            else:
                if (self.use_fwfm or self.use_fm) and require_update:
                    last_layer_size += (self.field_size + self.embedding_size)
                if self.use_deep and require_update:
                    last_layer_size += (self.deep_layers[-1] + 1)
                require_update = False
                if name in ['fm_1st.weight', 'fm_2nd.weight'] or 'fc.weight' in name:
                    param.data = TORCH.FloatTensor(param.data.size()).normal_().mul(np.sqrt(2.0 / last_layer_size))


    def fit(self, Xi_train, Xv_train, y_train, Xi_valid=None, Xv_valid=None, 
            y_valid = None, ealry_stopping=False, refit=False, save_path = None, prune=0, prune_fm=0, prune_r=0, prune_deep=0, emb_r=1., emb_corr=1.):
        if self.verbose:
            print("pre_process data ing...")
        is_valid = False
        Xi_train = np.array(Xi_train).reshape((-1, self.field_size-self.num, 1))
        Xv_train = np.array(Xv_train).reshape((-1, self.field_size-self.num, 1))
        y_train = np.array(y_train)
        x_size = Xi_train.shape[0]
        if Xi_valid:
            Xi_valid = np.array(Xi_valid).reshape((-1, self.field_size-self.num, 1))
            Xv_valid = np.array(Xv_valid)
            y_valid = np.array(y_valid)
            x_valid_size = Xi_valid.shape[0]
            is_valid = True
        if self.verbose:
            print("pre_process data finished")

        print('init_weights')
        self.init_weights()

        """
            train model
        """
        model = self.train()

        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=self.momentum, weight_decay=self.weight_decay)
        criterion = F.binary_cross_entropy_with_logits

        train_result = []
        valid_result = []
        num_total = 0
        num_1st_order_embeddings = 0
        num_2nd_order_embeddings = 0
        num_dnn = 0
        print('========')
        for name, param in model.named_parameters():
            print(name, param.data.shape)
            num_total += np.prod(param.data.shape)
            if '1st_embeddings' in name:
                num_1st_order_embeddings += np.prod(param.data.shape)
            if '2nd_embeddings' in name:
                num_2nd_order_embeddings += np.prod(param.data.shape)
            if 'linear_' in name:
                num_dnn += np.prod(param.data.shape)
        print('Summation of feature sizes: %s' % (sum(self.feature_sizes)))
        print('Number of 1st order embeddings: %d' % (num_1st_order_embeddings))
        print('Number of 2nd order embeddings: %d' % (num_2nd_order_embeddings))
        print('Number of DNN parameters: %d' % (num_dnn))
        print("Number of total parameters: %d"% (num_total))
        n_iter = 0
        for epoch in range(self.n_epochs):
            total_loss = 0.0
            batch_iter = x_size // self.batch_size
            epoch_begin_time = time()
            batch_begin_time = time()
            for i in range(batch_iter+1):
                if epoch >= self.warm:
                    n_iter += 1
                offset = i*self.batch_size
                end = min(x_size, offset+self.batch_size)
                if offset == end:
                    break
                batch_xi = Variable(torch.LongTensor(Xi_train[offset:end]))
                batch_xv = Variable(torch.FloatTensor(Xv_train[offset:end]))
                batch_y = Variable(torch.FloatTensor(y_train[offset:end]))
                if self.use_cuda:
                    batch_xi, batch_xv, batch_y = batch_xi.cuda(), batch_xv.cuda(), batch_y.cuda()
                optimizer.zero_grad()
                outputs = model(batch_xi, batch_xv)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()

                total_loss += loss.data.item()
                if self.verbose and i % 100 == 99:
                    eval = self.evaluate(batch_xi, batch_xv, batch_y)
                    print('[%d, %5d] loss: %.6f metric: %.6f time: %.1f s' %
                          (epoch + 1, i + 1, total_loss/100.0, eval, time()-batch_begin_time))
                    total_loss = 0.0
                    batch_begin_time = time()

                if prune and (i == batch_iter or i % 10 == 9) and epoch >= self.warm:
                    self.adaptive_sparse = self.target_sparse * (1 - 0.99**(n_iter /100.))
                    if prune_fm != 0:
                        stacked_embeddings = []
                        for name, param in model.named_parameters():
                            if 'fm_2nd_embeddings' in name:
                                stacked_embeddings.append(param.data)
                        stacked_emb = torch.cat(stacked_embeddings, 0)
                        emb_threshold = self.binary_search_threshold(stacked_emb.data, self.adaptive_sparse * emb_r, np.prod(stacked_emb.data.shape))
                    for name, param in model.named_parameters():
                        if 'fm_2nd_embeddings' in name and prune_fm != 0:
                            mask = abs(param.data) < emb_threshold
                            param.data[mask] = 0
                        if 'linear' in name and 'weight' in name and prune_deep != 0:
                            layer_pars = np.prod(param.data.shape)
                            threshold = self.binary_search_threshold(param.data, self.adaptive_sparse, layer_pars)
                            mask = abs(param.data) < threshold
                            param.data[mask] = 0 
                        if 'field_cov.weight' == name and prune_r != 0:
                            layer_pars = np.prod(param.data.shape)
                            symm_sum = 0.5 * (param.data + param.data.t())
                            threshold = self.binary_search_threshold(symm_sum, self.adaptive_sparse * emb_corr, layer_pars)
                            mask = abs(symm_sum) < threshold
                            param.data[mask] = 0
                            #print (mask.sum().item(), layer_pars)

                            

            no_non_sparse = 0
            for name, param in model.named_parameters():
                no_non_sparse += (param != 0).sum().item()
            print('Model parameters %d, sparse rate %.2f%%' % (no_non_sparse, 100 - no_non_sparse * 100. / num_total))
            train_loss, train_eval, preds = self.eval_by_batch(Xi_train,Xv_train,y_train,x_size)
            train_result.append(train_eval)
            print('Training [%d] loss: %.6f metric: %.6f sparse %.2f%% time: %.1f s' %
                  (epoch + 1, train_loss, train_eval, 100 - no_non_sparse * 100. / num_total, time()-epoch_begin_time))
            if is_valid:
                valid_loss, valid_eval, preds = self.eval_by_batch(Xi_valid, Xv_valid, y_valid, x_valid_size)
                valid_result.append(valid_eval)
                print('Validation [%d] loss: %.6f metric: %.6f sparse %.2f%% time: %.1f s' %
                      (epoch + 1, valid_loss, valid_eval, 100 - no_non_sparse * 100. / num_total, time()-epoch_begin_time))
            print('*' * 50)
            
            permute_idx = np.random.permutation(x_size)
            Xi_train = Xi_train[permute_idx]
            Xv_train = Xv_train[permute_idx]
            y_train = y_train[permute_idx]
            print('Training dataset shuffled.')
            
            if save_path:
                torch.save(self.state_dict(),save_path)
            if is_valid and ealry_stopping and self.training_termination(valid_result):
                print("early stop at [%d] epoch!" % (epoch+1))
                break
        num_total = 0
        num_1st_order_embeddings = 0
        num_2nd_order_embeddings = 0
        num_dnn = 0
        print('========')
        for name, param in model.named_parameters():
            num_total += (param != 0).sum().item()
            if '1st_embeddings' in name:
                num_1st_order_embeddings += (param != 0).sum().item()
            if '2nd_embeddings' in name:
                num_2nd_order_embeddings += (param != 0).sum().item()
            if 'linear_' in name:
                num_dnn += (param != 0).sum().item()
            if 'field_cov.weight' == name:
                symm_sum = 0.5 * (param.data + param.data.t())
                non_zero_r = (symm_sum != 0).sum().item()
        print('Number of pruned 1st order embeddings: %d' % (num_1st_order_embeddings))
        print('Number of pruned 2nd order embeddings: %d' % (num_2nd_order_embeddings))
        print('Number of pruned 2nd order interactions: %d' % (non_zero_r))
        print('Number of pruned DNN parameters: %d' % (num_dnn))
        print("Number of pruned total parameters: %d"% (num_total))
       

    def eval_by_batch(self,Xi, Xv, y, x_size):
        total_loss = 0.0
        y_pred = []
        if self.use_ffm:
            batch_size = 8192*2
        else:
            batch_size = 8192
        batch_iter = x_size // batch_size
        criterion = F.binary_cross_entropy_with_logits
        model = self.eval()
        for i in range(batch_iter+1):
            offset = i * batch_size
            end = min(x_size, offset + batch_size)
            if offset == end:
                break
            batch_xi = Variable(torch.LongTensor(Xi[offset:end]))
            batch_xv = Variable(torch.FloatTensor(Xv[offset:end]))
            batch_y = Variable(torch.FloatTensor(y[offset:end]))
            if self.use_cuda:
                batch_xi, batch_xv, batch_y = batch_xi.cuda(), batch_xv.cuda(), batch_y.cuda()
            outputs = model(batch_xi, batch_xv)
            pred = torch.sigmoid(outputs).cpu()
            y_pred.extend(pred.data.numpy())
            loss = criterion(outputs, batch_y)
            total_loss += loss.data.item()*(end-offset)
        total_metric = self.eval_metric(y,y_pred)
        return total_loss/x_size, total_metric, y_pred

    def binary_search_threshold(self, param, target_percent, total_no):
        l, r= 0., 1e2
        cnt = 0
        while l < r:
            cnt += 1
            mid = (l + r) / 2
            sparse_items = (abs(param) < mid).sum().item() * 1.0
            sparse_rate = sparse_items / total_no
            if abs(sparse_rate - target_percent) < 0.0001:
                return mid
            elif sparse_rate > target_percent:
                r = mid
            else:
                l = mid
            if cnt > 100:
                break
        return mid
    
    # shuffle three lists simutaneously
    def shuffle_in_unison_scary(self, a, b, c):
        rng_state = np.random.get_state()
        np.random.shuffle(a)
        np.random.set_state(rng_state)
        np.random.shuffle(b)
        np.random.set_state(rng_state)
        np.random.shuffle(c)

    def training_termination(self, valid_result):
        if len(valid_result) > 4:
            if self.greater_is_better:
                if valid_result[-1] < valid_result[-2] and \
                    valid_result[-2] < valid_result[-3] and \
                    valid_result[-3] < valid_result[-4]:
                    return True
            else:
                if valid_result[-1] > valid_result[-2] and \
                    valid_result[-2] > valid_result[-3] and \
                    valid_result[-3] > valid_result[-4]:
                    return True
        return False


    def predict(self, Xi, Xv):
        Xi = np.array(Xi).reshape((-1,self.field_size,1))
        Xi = Variable(torch.LongTensor(Xi))
        Xv = Variable(torch.FloatTensor(Xv))
        if self.use_cuda and torch.cuda.is_available():
            Xi, Xv = Xi.cuda(), Xv.cuda()

        model = self.eval()
        pred = torch.sigmoid(model(Xi, Xv.reshape((-1,10,1)))).cpu()
        return (pred.data.numpy() > 0.5)

    def predict_proba(self, Xi, Xv):
        Xi = np.array(Xi).reshape((-1, self.field_size, 1))
        Xi = Variable(torch.LongTensor(Xi))
        Xv = Variable(torch.FloatTensor(Xv))
        if self.use_cuda and torch.cuda.is_available():
            Xi, Xv = Xi.cuda(), Xv.cuda()

        model = self.eval()
        pred = torch.sigmoid(model(Xi, Xv.reshape((-1,10,1)))).cpu()
        return pred.data.numpy()

    def inner_predict(self, Xi, Xv):
        model = self.eval()
        pred = torch.sigmoid(model(Xi, Xv)).cpu()
        return (pred.data.numpy() > 0.5)

    def inner_predict_proba(self, Xi, Xv):
        model = self.eval()
        pred = torch.sigmoid(model(Xi, Xv)).cpu()
        return pred.data.numpy()

    def evaluate(self, Xi, Xv, y):
        y_pred = self.inner_predict_proba(Xi, Xv)
        return self.eval_metric(y.cpu().data.numpy(), y_pred)

# **TRAIN-FWFM**

In [192]:
def transform_index (X):
    X_indices = X.indices.reshape((-1,10)).copy()
    for i in range(10):
         X_indices[:,i] -= feature_sizes[0:i].sum()
    return Variable(torch.LongTensor(X_indices))   

In [193]:
X_indices = transform_index(X)

In [209]:
model = FWFM(10, feature_sizes, embedding_size = 10, use_fwfm= True, batch_size = 200, n_epochs= 15, learning_rate=0.01)

Cuda is not available, automatically changed into cpu model
The model is fwfm only
Init fwfm part


In [210]:
model.fit(X_indices, X.data, Y)

init_weights
bias torch.Size([1])
fm_1st_embeddings.0.weight torch.Size([7, 1])
fm_1st_embeddings.1.weight torch.Size([24, 1])
fm_1st_embeddings.2.weight torch.Size([213, 1])
fm_1st_embeddings.3.weight torch.Size([528, 1])
fm_1st_embeddings.4.weight torch.Size([3256, 1])
fm_1st_embeddings.5.weight torch.Size([782, 1])
fm_1st_embeddings.6.weight torch.Size([1209, 1])
fm_1st_embeddings.7.weight torch.Size([8, 1])
fm_1st_embeddings.8.weight torch.Size([80, 1])
fm_1st_embeddings.9.weight torch.Size([2893, 1])
fm_2nd_embeddings.0.weight torch.Size([7, 10])
fm_2nd_embeddings.1.weight torch.Size([24, 10])
fm_2nd_embeddings.2.weight torch.Size([213, 10])
fm_2nd_embeddings.3.weight torch.Size([528, 10])
fm_2nd_embeddings.4.weight torch.Size([3256, 10])
fm_2nd_embeddings.5.weight torch.Size([782, 10])
fm_2nd_embeddings.6.weight torch.Size([1209, 10])
fm_2nd_embeddings.7.weight torch.Size([8, 10])
fm_2nd_embeddings.8.weight torch.Size([80, 10])
fm_2nd_embeddings.9.weight torch.Size([2893, 10])
fm

In [211]:
train_indices = transform_index(train_one_hot)

In [212]:
valid_size = y_train.shape[0]

In [213]:
loss, auc_score, y_pred = model.eval_by_batch(train_indices.reshape((-1,10,1))[0:valid_size], train_one_hot.data.reshape((-1,10,1))[0:valid_size], y_train.to_numpy()[0:valid_size], valid_size)

In [214]:
y_out = np.array(y_pred) > 0.5

In [215]:
##ALL DATA
print(f'log_loss:  {loss}')
print(f'auc_score: {auc_score}')
print(f'f1_score: {f1_score(y_train[0:valid_size], y_out)}')

log_loss:  0.6492945670835558
auc_score: 0.661612585732263
f1_score: 0.4177112760686477


array([ True,  True, False, ...,  True,  True,  True])

In [216]:
import dill
filename = '/content/drive/MyDrive/ML_Project/FWFM/fmwm2.pickle'
dill.dump(model, open(filename, 'wb'))

In [ ]:
###### predictions

In [217]:
test_one_hot = enc.transform(test.to_numpy())

In [218]:
test_indices = transform_index(test_one_hot)

In [219]:
valid_size = test.shape[0]

In [220]:
loss, auc_score, y_pred = model.eval_by_batch(test_indices.reshape((-1,10,1))[0:valid_size], test_one_hot.data.reshape((-1,10,1))[0:valid_size], y_train.to_numpy()[0:valid_size], valid_size)

In [221]:
y_out = np.array(y_pred) > 0.5

In [222]:
np.save('/content/drive/MyDrive/ML_Project/FWFM/y_pred',y_out)

# **Report-FWFM**

<div dir = 'rtl'>

در این روش مشابه حالت FM کار‌های ابتدایی را انجام دادیم و لذا چیزی عوض نشده است. Fwfm یک حالت ساده‌تری از ffm است که تعداد پارامتر‌های کمتری دارد و از این جهت بهتر است.
 اما برای پیاده‌سازی نیازی به استفاده از pytroch داشتیم که بتواند گرادیان‌هارو به طرز مناسب‌تری حساب کنند و لذا از این فریمورک استفاده کردیم و همانطور که در کد مشاهده می‌شود. پیاده‌سازی صورت گرفته منعطف است و می‌توان از روش‌هایی مثل weightdecay استفاده نمود.


در مورد نتیجه نیز از همه‌ی مدل‌هایی که داشتیم نتیجه بهتر شد و این از بیش‌ نیز قابل پیش‌بینی بود.
</div>